# 개인화 추천을 위한 Amazon Bedrock AgentCore Memory

## 개요

이 튜토리얼에서는 [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html) namespace를 사용하여 개인화된 고객 경험을 제공하는 방법을 살펴봅니다. 추천, 거래, 선호도, 생애 event 등 여러 namespace 유형을 동적으로 질의하고 이를 종합적으로 추론하여 컨텍스트에 맞는 개인화된 응답을 제공하는 Agent를 구축합니다. Namespace를 사용하는 Memory는 Agent에 실시간 개인화를 위한 기본 컨텍스트 데이터 액세스를 제공합니다.

### Namespace를 사용하는 Memory가 필요한 이유

기존 database, RAG 시스템 또는 DynamoDB 같은 key-value 저장소 대신 AgentCore Memory namespace를 사용하는 이유는 무엇일까요?

**Namespace path는 Agent에 최적화되어 있습니다.**
- 계층적이고 의미가 명확하며 조합 가능 - `/bank/customers/{id}/recommendations/pending` 자체로 의미를 설명
- Agent가 SQL 지식 없이 대화 컨텍스트에서 path를 동적으로 구성
- schema 설계, join, connection pooling 또는 migration이 필요 없음

**결정적 검색과 fuzzy search 비교:**
- 정확한 namespace 일치는 필요한 내용을 정확히 반환하며 relevance scoring이나 semantic drift가 없음
- RAG는 비정형 지식에 적합하지만 비즈니스 프로세스 데이터에는 정밀성이 필요

**기본 제공 actor 격리:**
- 고객 데이터가 namespace path에 따라 자연스럽게 분할
- 고객 간 데이터 유출 위험이 없음

**동적 다중 소스 추론:**
- Agent가 질문에 따라 질의할 namespace와 순서를 결정
- 고정된 query plan 없이 LLM이 여러 namespace 유형의 컨텍스트를 구성
- 새 데이터 소스(예: `/complaints`) 추가 시 schema 변경 없이 쓰기만 하면 됨


### 튜토리얼 세부 정보

| 정보         | 세부 정보                                                    |
|:--------------------|:-----------------------------------------------------------|
| 튜토리얼 유형       | 개인화 추천을 위한 Memory                    |
| 기능             | 장기 메모리 Namespace + Strands Agent                |
| 주요 기능        | 다중 Namespace 추론, 동적 코드 생성, Agent State |
| 예제 난이도  | 중급                                               |
| 사용 SDK            | boto3, bedrock-agentcore, strands-agents, strands-agents-tools |

### 학습 내용

1. Memory 인스턴스를 생성하고 여러 데이터 유형(추천, 거래, 선호도, 생애 event) 입력
2. Agent 추론에 따라 모든 namespace를 동적으로 질의하는 도구 구축
3. 하드 코딩하지 않고 Agent 상태를 통해 고객 컨텍스트를 결정적으로 전달
4. Agent가 여러 namespace 유형을 종합적으로 추론하여 복잡한 질문에 답하도록 구성
5. 동적 계산에 `python_repl` 사용(예: 지출과 카드 혜택 비교)

### 아키텍처

![아키텍처](architecture.png)

### 작동 방식

Agent는 고객별로 여러 namespace 유형에 액세스할 수 있습니다.

```
/bank/customers/{id}/recommendations/{stage}  → pending, shown, declined recommendations
/bank/customers/{id}/transactions/summary     → monthly spending by category
/bank/customers/{id}/preferences              → stated preferences from conversations
/bank/customers/{id}/life_events              → detected life events (travel plans, etc.)
```

고객이 "어떤 카드를 발급받아야 하나요?"라고 질문하면 Agent는 다음을 수행합니다.
1. `recommendations/pending`을 질의하여 사용 가능한 추천 확인
2. `transactions/summary`를 질의하여 실제 지출과 비교 검증
3. `preferences`를 질의하여 충돌 여부 확인(예: "연회비 없음")
4. `python_repl`로 잠재적 절감액 계산

이러한 다중 namespace 계층 구조와 LLM 추론을 통해 Agent는 각 질의에 맞는 컨텍스트를 *동적으로* 구성하여 복잡한 질의에 답할 수 있습니다.

**→ 고객 간 분석과 marketing insight는 Part 2에서 계속합니다.**

## 0. 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* AgentCore Memory 및 Amazon Bedrock 액세스 권한이 구성된 AWS 자격 증명
* Amazon Bedrock 모델 액세스(Claude Sonnet)

먼저 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install "boto3>=1.42.63" "bedrock-agentcore[strands-agents]" strands-agents strands-agents-tools

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성합니다.

In [ ]:
import json
import boto3
import time
import uuid
from datetime import datetime
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory.client import MemoryClient
from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands_tools import python_repl

import os

os.environ["BYPASS_TOOL_CONSENT"] = "True"
os.environ["PYTHON_REPL_INTERACTIVE"] = "false"


# 구성
REGION = "us-west-2"
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

# Client 초기화
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)
memory_client = MemoryClient(region_name=REGION)

print(f"✅ Initialized clients for region: {REGION}")

## 1. Memory 인스턴스 생성

Strategy 없이 Memory 인스턴스를 생성합니다. 대화에서 추출하는 대신 여러 소스의 외부 데이터로 채웁니다.

In [ ]:
unique_name = f"personalisation_demo_{uuid.uuid4().hex[:8]}"
memory = memory_client.create_memory_and_wait(
    name=unique_name,
    strategies=[],
    description="Demo memory for personalised credit card recommendations",
)

MEMORY_ID = memory["memoryId"]
print(f"✅ Created memory: {MEMORY_ID}")

## 2. 다중 소스 샘플 데이터 입력

프로덕션에서는 다음과 같은 여러 시스템에서 이 데이터를 가져옵니다.
- 추천 엔진 또는 CRM의 **추천**
- core banking 또는 카드 처리 시스템의 **거래**
- 과거 대화 또는 명시적 설정에서 추출한 **선호도**
- 거래 pattern 또는 고객 상호작용에서 감지한 **생애 event**

핵심은 AgentCore Memory가 이 모든 *외부 데이터*를 Agent 상태에 통합할 수 있는 단일 namespace 구조를 제공한다는 점입니다. Agent는 기반 소스 시스템을 몰라도 이러한 데이터를 자연스럽게 질의할 수 있습니다.

In [ ]:
# 고객 001: 여행을 자주 하며 가격에 민감
customer_001_data = [
    # 추천
    {
        "namespace": "/bank/customers/customer_001/recommendations/pending",
        "content": {
            "product": "Travel Rewards Card",
            "annual_fee": 95,
            "benefits": "3x points on travel, 2x dining",
            "reason": "High travel spending detected",
        },
    },
    {
        "namespace": "/bank/customers/customer_001/recommendations/declined",
        "content": {
            "product": "Premium Platinum Card",
            "annual_fee": 495,
            "declined_reason": "Customer objected to annual fee",
            "declined_date": "2024-01-15",
        },
    },
    # 거래 요약
    {
        "namespace": "/bank/customers/customer_001/transactions/summary",
        "content": {
            "monthly_avg": {
                "travel": 2400,
                "dining": 800,
                "groceries": 600,
                "gas": 200,
                "other": 500,
            },
            "total_monthly": 4500,
            "period": "last_6_months",
        },
    },
    # 선호도
    {
        "namespace": "/bank/customers/customer_001/preferences",
        "content": {
            "max_annual_fee": 100,
            "priority_categories": ["travel", "dining"],
            "stated": "Prefers cards with low annual fees but good travel benefits",
        },
    },
    # 생애 event
    {
        "namespace": "/bank/customers/customer_001/life_events",
        "content": {
            "events": [
                {
                    "type": "upcoming_travel",
                    "details": "Japan trip booked for next month",
                    "detected": "2024-02-01",
                },
                {
                    "type": "spending_increase",
                    "category": "travel",
                    "change": "+40%",
                    "detected": "2024-01-15",
                },
            ]
        },
    },
]

# 고객 002: 고소득이며 premium 혜택을 중시
customer_002_data = [
    {
        "namespace": "/bank/customers/customer_002/recommendations/pending",
        "content": {
            "product": "Premium Platinum Card",
            "annual_fee": 495,
            "benefits": "5x all travel, lounge access, $300 travel credit",
            "reason": "High income segment, luxury spending patterns",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/recommendations/accepted",
        "content": {
            "product": "Business Rewards",
            "annual_fee": 150,
            "accepted_date": "2024-01-20",
            "reason": "Business expenses detected",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/transactions/summary",
        "content": {
            "monthly_avg": {
                "travel": 5000,
                "dining": 2000,
                "luxury": 3000,
                "business": 4000,
                "other": 1000,
            },
            "total_monthly": 15000,
            "period": "last_6_months",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/preferences",
        "content": {
            "max_annual_fee": "no_limit",
            "priority_categories": ["travel", "luxury", "business"],
            "stated": "Values premium benefits and status, fee is not a concern",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/life_events",
        "content": {
            "events": [
                {
                    "type": "business_growth",
                    "details": "Business expenses up 60%",
                    "detected": "2024-02-01",
                }
            ]
        },
    },
]

# 고객 003: 학생이며 가격에 매우 민감
customer_003_data = [
    {
        "namespace": "/bank/customers/customer_003/recommendations/pending",
        "content": {
            "product": "No-Fee Starter Card",
            "annual_fee": 0,
            "benefits": "1% cashback on everything",
            "reason": "Alternative after fee objection",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/recommendations/declined",
        "content": {
            "product": "Student Card",
            "annual_fee": 95,
            "declined_reason": "Annual fee too high",
            "declined_date": "2024-02-01",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/transactions/summary",
        "content": {
            "monthly_avg": {
                "groceries": 300,
                "dining": 150,
                "entertainment": 100,
                "transport": 80,
                "other": 70,
            },
            "total_monthly": 700,
            "period": "last_6_months",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/preferences",
        "content": {
            "max_annual_fee": 0,
            "priority_categories": ["groceries", "dining"],
            "stated": "Student budget, absolutely no annual fees",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/life_events",
        "content": {
            "events": [
                {
                    "type": "student",
                    "details": "University student",
                    "detected": "2023-09-01",
                }
            ]
        },
    },
]

all_data = customer_001_data + customer_002_data + customer_003_data
print(f"📊 Prepared {len(all_data)} records across 3 customers and 4 namespace types")

In [ ]:
records = []
current_time = datetime.now().timestamp()

for idx, item in enumerate(all_data):
    records.append(
        {
            "requestIdentifier": f"record_{idx:03d}",
            "namespaces": [item["namespace"]],
            "content": {"text": json.dumps(item["content"])},
            "timestamp": current_time + idx,
        }
    )

response = agentcore_client.batch_create_memory_records(memoryId=MEMORY_ID, records=records)

print(f"✅ Created {len(response['successfulRecords'])} records")
print("   Namespace types: recommendations, transactions/summary, preferences, life_events")

## 3. Agent Tool 정의

모든 namespace path를 질의할 수 있는 유연한 단일 도구를 생성합니다. Agent는 다음을 결정합니다.
- 질의할 namespace 유형(recommendations, transactions, preferences, life_events)
- 사용할 하위 path(예: `recommendations/pending` 또는 `recommendations/declined`)

이는 SQL과 근본적으로 다릅니다. 고정된 schema나 query plan 없이 Agent가 대화에 따라 namespace path를 동적으로 구성합니다.

In [ ]:
time.sleep(120)
session_manager = MemorySessionManager(memory_id=MEMORY_ID, region_name=REGION)

from strands.types.tools import ToolContext


@tool(context=True)
def query_customer_memory(namespace_type: str, tool_context: ToolContext) -> str:
    """Query customer data from a specific namespace type.

    Args:
        namespace_type: The type of data to query. Options:
            - 'recommendations/pending' - pending card recommendations
            - 'recommendations/declined' - previously declined recommendations
            - 'recommendations/accepted' - accepted recommendations
            - 'transactions/summary' - monthly spending by category
            - 'preferences' - customer stated preferences
            - 'life_events' - detected life events and changes
            - 'all' - query all data for the customer
        tool_context: Provides access to customer_id via invocation_state.

    Returns:
        JSON string containing the matching data.
    """
    customer_id = tool_context.invocation_state.get("customer_id")
    if not customer_id:
        return json.dumps({"error": "No customer_id in context"})

    if namespace_type == "all":
        namespace_prefix = f"/bank/customers/{customer_id}/"
    else:
        ns = namespace_type.rstrip("/")
        namespace_prefix = f"/bank/customers/{customer_id}/{ns}"

    records = session_manager.list_long_term_memory_records(namespace_prefix=namespace_prefix, max_results=100)

    results = [json.loads(r["content"]["text"]) for r in records]
    return json.dumps(
        {
            "customer_id": customer_id,
            "namespace_queried": namespace_type,
            "count": len(results),
            "data": results,
        },
        indent=2,
    )


print("✅ Defined query_customer_memory tool")

## 4. 개인화 Agent 생성

Agent에는 다음 도구가 있습니다.
- `query_customer_memory` - 모든 namespace 유형에서 데이터 가져오기
- `python_repl` - 계산 수행(예: 카드 혜택의 잠재적 절감액)

System prompt는 Agent가 여러 데이터 소스를 종합적으로 추론하도록 안내합니다.

In [ ]:
SYSTEM_PROMPT = """You are a personalised banking assistant. You help customers understand their credit card options and make informed decisions.

You have access to multiple types of customer data via query_customer_memory:
- recommendations/pending, recommendations/declined, recommendations/accepted
- transactions/summary (monthly spending by category)
- preferences (stated preferences like max annual fee)
- life_events (detected events like upcoming travel)

When answering questions:
1. Query the relevant namespace(s) to gather context
2. Cross-reference data sources (e.g., check if a recommendation aligns with preferences)
3. Use python_repl for calculations when needed (e.g., potential rewards based on spending)
4. Explain your reasoning in customer-friendly terms

The customer's identity is already known - you don't need to ask for it.
"""

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)


def create_agent():
    """대화 기록이 없는 새 에이전트 인스턴스를 생성합니다."""
    return Agent(
        model=model,
        tools=[query_customer_memory, python_repl],
        system_prompt=SYSTEM_PROMPT,
        callback_handler=None,
    )


agent = create_agent()
print("\u2705 Created personalisation agent")

## 5. 다중 Namespace 추론 데모

이제 Agent가 여러 데이터 소스를 종합적으로 추론하는 과정을 살펴보겠습니다. 이러한 질의에서 Agent는 다음 작업을 수행해야 합니다.
1. 질의할 namespace 결정
2. 서로 다른 소스의 정보 상호 참조
3. 계산용 코드 생성

이는 단일 SQL 질의로 수행할 수 없습니다. Agent가 컨텍스트를 동적으로 구성합니다.

In [ ]:
# 고객 001: 여행을 자주 하며 가격에 민감
current_customer = "customer_001"
print(f"🧑 Chatting as: {current_customer}\n")
print("=" * 60)

# 질의 1: 추천과 선호도의 상호 참조 필요
response = agent(
    "What credit card do you recommend for me? Make sure it fits my preferences.",
    customer_id=current_customer,
)
print(f"\n💬 Response:\n{response}")

In [ ]:
# 질의 2: 거래 + 추천 + 계산 필요
response = agent(
    "How much would I earn in rewards with the Travel Rewards Card based on my actual spending?",
    customer_id=current_customer,
)
print(f"\n💬 Response:\n{response}")

In [ ]:
# 질의 4: 설명을 위해 거절 기록 + 선호도 필요
response = agent(
    "Why was I recommended the Travel Rewards Card instead of the Premium Platinum?",
    customer_id=current_customer,
)
print(f"\n💬 Response:\n{response}")

## 6. 고객 컨텍스트 전환

이전 고객의 대화 기록이 컨텍스트에 섞이지 않도록 고객마다 새 Agent 인스턴스를 사용합니다.

In [ ]:
# 고객 003: 학생이며 가격에 매우 민감
agent = create_agent()
current_customer = "customer_003"
print(f"\n\U0001f9d1 Now chatting as: {current_customer}\n")
print("=" * 60)

response = agent(
    "I'm a student on a tight budget. What card options do I have? I really can't afford any annual fees.",
    customer_id=current_customer,
)
print(f"\n\U0001f4ac Response:\n{response}")

In [ ]:
# 고객 002: 고소득이며 premium을 중시
agent = create_agent()
current_customer = "customer_002"
print(f"\n\U0001f9d1 Now chatting as: {current_customer}\n")
print("=" * 60)

response = agent(
    "I travel a lot for business. What's your best premium card? I don't care about the fee if the benefits are worth it.",
    customer_id=current_customer,
)
print(f"\n\U0001f4ac Response:\n{response}")

## 요약

### 구현 내용

1. **다중 namespace 추론** - Agent가 추천, 거래, 선호도, 생애 event를 질의하고 상호 참조하여 컨텍스트에 맞는 답변 제공

2. **동적 namespace 선택** - 고정된 query plan이 아니라 질문에 따라 Agent가 질의할 namespace를 결정

3. **계산용 코드 생성** - Agent가 `python_repl`을 사용하여 실제 지출 데이터를 기반으로 잠재적 reward 계산

4. **결정적 고객 컨텍스트** - 고객 ID가 `invocation_state`를 통해 전달되어 간편하게 컨텍스트 전환

### SQL로 수행할 수 없는 이유

- **고정된 query plan 없음** - 질문에 따라 Agent가 가져올 내용을 결정
- **소스 간 추론** - 추천, 선호도, 거래를 결합하려면 SQL join이 아닌 LLM 추론이 필요
- **의미가 있는 namespace path** - Agent가 path 이름에서 `/preferences`와 `/life_events`의 차이를 이해
- **schema 변경 불필요** - `/complaints`를 추가해도 migration 불필요

### 제공되는 기능

- 각 고객의 전체 컨텍스트에 적응하는 실시간 개인화 경험
- 다중 소스 비즈니스 데이터에 대한 자연어 질의
- schema 재설계 없이 새 데이터 소스를 간편하게 통합
- Agent에 최적화된 데이터 액세스 pattern

**→ 고객 간 분석, 제품 수준 insight, marketing 사용 사례는 Part 2에서 계속합니다.**

## 7. 리소스 정리(선택 사항)

실습을 마치면 이 튜토리얼에서 생성한 리소스를 정리합니다.

In [ ]:
try:
    memory_client.delete_memory_and_wait(memory_id=MEMORY_ID)
    print(f"✅ Deleted memory resource: {MEMORY_ID}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")